In [2]:

import pandas as pd

# Training dataset
data = {
    'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
    'AirTemp': ['Warm', 'Warm', 'Cold', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High'],
    'Wind': ['Strong', 'Strong', 'Weak', 'Strong'],
    'Water': ['Warm', 'Warm', 'Cool', 'Warm'],
    'Forecast': ['Same', 'Same', 'Change', 'Same'],
    'EnjoySport': ['Yes', 'Yes', 'No', 'Yes']
}

df = pd.DataFrame(data)

print("Training Dataset:")
print(df)

# FIND-S algorithm
attributes = list(df.columns[:-1])
target = 'EnjoySport'

hypothesis = ['0'] * len(attributes)

for _, row in df.iterrows():
    if row[target] == 'Yes':
        for j in range(len(attributes)):
            if hypothesis[j] == '0':
                hypothesis[j] = row[attributes[j]]
            elif hypothesis[j] != row[attributes[j]]:
                hypothesis[j] = '?'

print("\nFinal Most Specific Hypothesis:")
print(hypothesis)


Training Dataset:
     Sky AirTemp Humidity    Wind Water Forecast EnjoySport
0  Sunny    Warm   Normal  Strong  Warm     Same        Yes
1  Sunny    Warm     High  Strong  Warm     Same        Yes
2  Rainy    Cold     High    Weak  Cool   Change         No
3  Sunny    Warm     High  Strong  Warm     Same        Yes

Final Most Specific Hypothesis:
['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']


In [3]:

import pandas as pd

data = {
    'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
    'AirTemp': ['Warm', 'Warm', 'Cold', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High'],
    'Wind': ['Strong', 'Strong', 'Weak', 'Strong'],
    'Water': ['Warm', 'Warm', 'Cool', 'Warm'],
    'Forecast': ['Same', 'Same', 'Change', 'Same'],
    'EnjoySport': ['Yes', 'Yes', 'No', 'Yes']
}

df = pd.DataFrame(data)
df.to_csv('training_data.csv', index=False)

print("CSV file created successfully!")
print(df)


CSV file created successfully!
     Sky AirTemp Humidity    Wind Water Forecast EnjoySport
0  Sunny    Warm   Normal  Strong  Warm     Same        Yes
1  Sunny    Warm     High  Strong  Warm     Same        Yes
2  Rainy    Cold     High    Weak  Cool   Change         No
3  Sunny    Warm     High  Strong  Warm     Same        Yes


In [4]:

import pandas as pd
from itertools import product

# Read training data from CSV
df = pd.read_csv('training_data.csv')

attributes = list(df.columns[:-1])
target = df.columns[-1]

# All possible attribute values
domains = [df[attr].unique().tolist() for attr in attributes]

# Specific and general boundaries
S = ['0'] * len(attributes)
G = [['?'] * len(attributes)]

def covers(h, x):
    return all(a == '?' or a == v for a, v in zip(h, x))

def more_general(h1, h2):
    return all(a == '?' or a == b or b == '0'
               for a, b in zip(h1, h2))

for _, row in df.iterrows():
    x = [row[a] for a in attributes]
    label = row[target]

    if label == 'Yes':
        # Generalize S to cover positive example
        for i in range(len(S)):
            if S[i] == '0':
                S[i] = x[i]
            elif S[i] != x[i]:
                S[i] = '?'

        # Remove general hypotheses that do not cover x
        G = [g for g in G if covers(g, x)]

    else:
        # Remove hypotheses in G that cover negative example
        new_G = []

        for g in G:
            if not covers(g, x):
                new_G.append(g)
                continue

            # Minimal specializations of g
            for i in range(len(g)):
                if g[i] == '?':
                    for value in domains[i]:
                        if value != x[i]:
                            h = g.copy()
                            h[i] = value

                            if more_general(h, S):
                                new_G.append(h)

        G = new_G

    # Remove duplicate hypotheses
    unique_G = []
    for h in G:
        if h not in unique_G:
            unique_G.append(h)
    G = unique_G

print("Training Dataset:")
print(df)

print("\nSpecific Boundary (S):")
print(S)

print("\nGeneral Boundary (G):")
for h in G:
    print(h)


Training Dataset:
     Sky AirTemp Humidity    Wind Water Forecast EnjoySport
0  Sunny    Warm   Normal  Strong  Warm     Same        Yes
1  Sunny    Warm     High  Strong  Warm     Same        Yes
2  Rainy    Cold     High    Weak  Cool   Change         No
3  Sunny    Warm     High  Strong  Warm     Same        Yes

Specific Boundary (S):
['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']

General Boundary (G):
['Sunny', '?', '?', '?', '?', '?']
['?', 'Warm', '?', '?', '?', '?']
['?', '?', '?', 'Strong', '?', '?']
['?', '?', '?', '?', 'Warm', '?']
['?', '?', '?', '?', '?', 'Same']
